# Gold Table: gold.video_performance

## Objective

Create a business-ready fact table containing one record per YouTube video with performance metrics and derived KPIs.

## Source
- Silver Layer

## Grain
- One row per video

## Columns

| Column | Description |
|---------|-------------|
| video_id | Unique video identifier |
| title | Video title |
| published_date | Date the video was published |
| publish_year | Year extracted from published_date |
| publish_month | Month extracted from published_date |
| video_type | Short or Normal |
| duration | Standardized video duration |
| views | Total views |
| likes | Total likes |
| comments | Total comments |
| engagement_rate | ((Likes + Comments) / Views) × 100 |

## Business Purpose

- Analyze individual video performance.
- Support detailed reporting.
- Provide source data for dashboard KPIs.
- Enable filtering by publication date and video type.

In [0]:
%sql
create schema if not exists youtube.gold;

In [0]:
from pyspark.sql import functions as f 

In [0]:
silver_table="youtube.silver.yt_data"
gold_table="youtube.gold.video_performance"

In [0]:
video_performance_extracted_df = (
    spark.read.table(silver_table)
    .withColumn("published_date", f.to_date("published_at"))
    .withColumn("publish_year", f.year("published_at"))
    .withColumn("publish_month", f.month("published_at"))
    .withColumn(
        "engagement_rate",
        f.round(((f.col("likes") + f.col("comments")) / f.col("views")) * 100, 2),
    )
    .select(
        f.col("video_id"),
        f.col("title"),
        f.col("published_date"),
        f.col("publish_year"),
        f.col("publish_month"),
        f.col("video_type"),
        f.col("duration"),
        f.col("views"),
        f.col("likes"),
        f.col("comments"),
        f.col("engagement_rate")
    )
)

In [0]:
(
    video_performance_extracted_df.write.mode("overwrite").saveAsTable(gold_table)
)

In [0]:
%sql
select * from youtube.gold.video_performance